In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "sk-proj"

In [ ]:
!pip install -q langchain-openai langchain-core requests

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.chat_models import ChatOpenAI
from langchain_core.messages import HumanMessage
import requests

In [ ]:
# tool create
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers"""
    return a * b

In [ ]:
print(multiply.invoke({"a": 2, "b": 3}))

In [ ]:
multiply.name

In [ ]:
multiply.description

In [ ]:
multiply.args

# tool binding

In [ ]:
llm  = ChatOpenAI()

In [ ]:
llm_with_tools = llm.bind_tools([multiply])

tool calling


In [ ]:
llm_with_tools.invoke('Hi how are you')

In [ ]:
query = HumanMessage("can you multiply 3 with 10")

In [ ]:
messages = [query]

In [ ]:
messages

In [ ]:
result = llm_with_tools.invoke(messages)

In [ ]:
messages.append(result)

In [ ]:
messages

In [ ]:
result.tool_calls[0]

tool execution

In [ ]:
tool_result = multiply.invoke(result.tool_calls[0])

send to llm

In [ ]:
messages.append(tool_result)

In [ ]:
messages

In [ ]:
llm_with_tools.invoke(messages).content

#Currency Conversion tool

In [ ]:
# tool create

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  '''
  This function fetches the conversion factor  between a given base currency and target currency
  '''
  url = f'https://api.exchangerate-api.com/v4/latest/{base_currency}/{target_currency}'
  response = requests.get(url)
  return  response.json()

  @tool
  def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg])-> float:
    """
    given a currency conversion rate this function calculates the target currency value from a given base currency value
    """
    return base_currency_value * conversion_rate


In [ ]:
get_conversion_factor.invoke({"base_currency": "USD", "target_currency": "INR"})

In [ ]:
convert.invoke({'base_currency_value': 10, 'conversion_rate': 85.16})

# tool binding

In [ ]:
llm = ChatOpenAI()


In [ ]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [ ]:
messages = [HumanMessages("What is the conversion factor between USD and INR , and based on that can you convert 10 USD to INR")]

In [ ]:
messages

In [ ]:
ai_messages = llm_with_tools.invoke(messages)

In [ ]:
messages.append(ai_messages)

In [ ]:
ai_messages.tool_calls

In [ ]:
import json

for tool_call in ai_messages.tool_calls:
  # execute the 1st tool and get the value of conversion rate
 if tool_call['name'] == 'get_conversion_factor':
    tool_messages1 = get_conversion_factor.invoke(tool_call)
    #fetch this conversion rate
    conversion_rate = json.loads(tool_messages1.content)['conversion_rate']
    # append this tool message to messages list
    messages.append(tool_messages1)

# execute the 2nd tool using the conversion rate from tool 1
 if tool_call['name']  == 'convert':
  # fetch the current arg
  tool_call['args']['conversion_rate'] = conversion_rate
  tool_messages2 = convert.invoke(tool_call)
  messages.append(tool_messages2)


In [ ]:
messages

In [ ]:
llm_with_tools.invoke(messages).content